<a href="https://colab.research.google.com/github/cyberirishman/5-day-AI-Cyber/blob/main/Lab3c_First_Classifier_AuthLog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Day 2 · Lab 3c — Your First Classifier on Security Data

**AI for Cybersecurity Professionals · Day 2: AI for Defense**

You now have every skill this lab needs:

- **Lab 3** — you found attacks in the login log with **hand-written rules**.
- **Lab 3a** — you **cleaned** messy data and proved it changes the answer.
- **Lab 3b** — you **normalized** columns onto a common ruler and proved that changes it too.

Labs 3a and 3b used houses on purpose, so the only lesson was the technique. Now we bring the
technique home: **the same pipeline, on the real login log from Labs 2 and 3.**

### What we are actually trying to achieve

We are going to **decide whether each login attempt is an attack or benign**, and this time we
are not going to write the rule ourselves. We are going to give a **Decision Tree** a pile of
labelled examples and let it work out the rule.

Then we grade it — properly, on logins it has never seen — using the **confusion matrix** from
§2.7. That is the first time in this course a real model gets a real score.

Finally we put it **head-to-head against your Lab 3 hand-written rules** on the exact same
logins, and read the tree to see *what it actually learned*. That last part is the most
important thing in this lab, and it does not go the way you might expect.

### What you'll do in each step

| Step | What happens |
|---|---|
| **1** | Load the libraries, and meet the **Decision Tree** — the model we will use. |
| **2** | Load the login log **straight from GitHub** — nothing to upload. |
| **3** | Meet the **label**: why we are suddenly allowed to use the answer key. |
| **4** | **Feature engineering**: turn raw log lines into four numbers a model can learn from. |
| **5** | **Split** the log: rows to learn from, rows held back for the exam. |
| **6** | Add the fifth feature — the one that looks *across* rows — without cheating. |
| **7** | **Normalize** to 0–1 (and an honest note about why a tree doesn't care). |
| **8** | **Train** the tree — and **draw it**, so you can read the flow-chart it invented. |
| **9** | **Score** it on the untouched test rows: the §2.7 confusion matrix, for real. |
| **10** | **Head-to-head** against your Lab 3 hand-written rules. |
| **11** | **Tune** it to catch more attacks — and watch the tree redraw itself. |
| **12** | **Read the tree honestly**: what did it *really* learn? |

> **Nothing to download or upload.** Open this notebook from its **Open in Colab** badge above
> and the data loads itself from GitHub.

> No prior Python needed — every block of code is explained in the comments (the grey text
> after a `#`). Python ignores those; they are notes for humans.

## Step 1 — Set up our tools

In [ ]:
# ---- The usual toolboxes -------------------------------------------------------
#   pandas     : tables (a table = a "DataFrame")
#   numpy      : fast maths on numbers
#   matplotlib : draws charts
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- The model we will use: a DECISION TREE -------------------------------------
#
# WHAT A DECISION TREE IS, IN ONE SENTENCE:
#   It is a flow-chart of yes/no questions. To judge a login it starts at the top
#   question, follows the yes or no branch, asks the next question, and keeps going
#   until it runs out of questions. Wherever it stops is its answer: attack, or benign.
#
# WHY THIS IS DIFFERENT FROM LAB 3:
#   In Lab 3 YOU wrote the questions ("more than 5 failures from one IP?").
#   Here the tree WRITES ITS OWN QUESTIONS by looking at labelled examples. That is
#   the whole point of machine learning, and the whole point of this lab.
#
# HOW IT CHOOSES A QUESTION (the honest version):
#   At every point it tries every feature and every possible cut-off, and keeps the
#   one that best separates attacks from benign logins. Then it repeats on each side.
#   That is all "training" means for a tree -- there is no magic in it.
#
# WHY WE PICKED A TREE FOR YOUR FIRST MODEL:
#   Because you can LOOK AT IT. We will literally draw the flow-chart it invented.
#   That is called a "glass box" model, and it is why analysts trust trees (-> §2.10).
from sklearn.tree import DecisionTreeClassifier   # the model itself
from sklearn.tree import plot_tree                # draws the flow-chart as a picture
from sklearn.tree import export_text              # prints the flow-chart as plain text

# ---- Splitting the data into "learn from" and "tested on" ------------------------
# Same idea as Lab 3b: a model graded on rows it memorised is marking its own homework.
from sklearn.model_selection import train_test_split

# ---- The normalizer from Lab 3b --------------------------------------------------
# MinMaxScaler rewrites every value as (value - min) / (max - min), so the smallest
# value in a column becomes 0 and the largest becomes 1.
from sklearn.preprocessing import MinMaxScaler

# ---- The scoreboard from section 2.7 ---------------------------------------------
#   confusion_matrix        : the 2x2 grid of TP / TN / FP / FN
#   ConfusionMatrixDisplay  : draws that grid as a picture
#   precision_score         : of the alerts we raised, how many were real?
#   recall_score            : of the real attacks, how many did we catch?
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score

# Let pandas print wide tables without hiding columns behind "..."
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Libraries loaded. Ready to go.")

**What you should see:** `Libraries loaded. Ready to go.` — and nothing else. If you get a red
error box here, run the cell again; on Colab the very first cell sometimes needs a second go.

## Step 2 — Load the login log

This is the **same** `day2_auth_logs.csv` you inspected by eye in Lab 2 and analysed with
hand-written rules in Lab 3. Nothing about the data has changed — only what we do with it.

We read it **straight from the internet**, so there is nothing to upload and no file path to
get wrong on Mac, Windows or Linux.

In [ ]:
# ---- Where the data comes from ---------------------------------------------------
# Note this is the RAW GitHub address (raw.githubusercontent.com), which returns the
# file itself. A normal github.com link returns a web PAGE and pandas cannot read it.
DATA_URL  = "https://raw.githubusercontent.com/cyberirishman/5-day-AI-Cyber/main/day2_auth_logs.csv"
DATA_FILE = "day2_auth_logs.csv"     # only used by the offline fallbacks below

import os   # lets us ask the computer whether a file exists on disk

# ---- Same loader as Labs 3a and 3b: internet first, then local file, then upload box ----
def load_csv(url="", fname=""):
    """Load the CSV from a URL, or a local file, or (on Colab) an upload box."""
    # parse_dates= tells pandas that this column holds date-and-time values, not text,
    # so we can later ask it questions like "what hour of the day was this?"
    if url:
        try:
            return pd.read_csv(url, parse_dates=["Login Timestamp"])
        except Exception as problem:
            print("Could not read from the internet:", problem)
            print("Falling back to a local copy...")
    for path in [fname, os.path.join("data", fname)]:
        if fname and os.path.exists(path):
            return pd.read_csv(path, parse_dates=["Login Timestamp"])
    # Last resort, Colab only: pop up a file picker.
    from google.colab import files
    up = files.upload()
    return pd.read_csv(list(up.keys())[0], parse_dates=["Login Timestamp"])

logs = load_csv(url=DATA_URL, fname=DATA_FILE)

print("Loaded {:,} login events.".format(len(logs)))
print("\nThe columns we have to work with:")
print(list(logs.columns))

**What you should see:** `Loaded 8,945 login events.` followed by the column list — the same
columns you read by eye in Lab 2: timestamp, username, IP address, country, user-agent string,
whether the login succeeded, and the two answer-key columns.

## Step 3 — Meet the label: why we are suddenly allowed to use the answer key

In Labs 2 and 3 there was a hard rule about two columns:

> `Is Attack IP` and `Is Account Takeover` are an **analyst answer key** — for checking your
> work, **never** used to detect.

**In this lab we break that rule on purpose.** Here is why that is not cheating.

A Decision Tree does not invent the idea of "attack" out of thin air. It learns by being shown
**examples that are already labelled** — *this login was an attack, that one wasn't* — and
working out what separates the two groups. That is what the word **supervised** means in
"supervised learning": something supervised the answers.

So the answer key changes job:

| Lab | `Is Attack IP` is… | Allowed? |
|---|---|---|
| Labs 2 & 3 | the marking scheme for a rule **you** wrote | look at it only *after* you answer |
| **Lab 3c (here)** | the **teacher** the model learns from | yes — a model cannot learn without it |
| Any lab | an input the model reads *at detection time* | **never** — that is cheating, and it is called *leakage* |

The catch — and this is the real-world question the exercise is teaching you — is:

> **Where do the labels come from when nobody hands you a CSV with the answers in it?**

They come from work: a confirmed incident, an analyst's verdict on a closed ticket, a threat-intel
feed, a customer telling you their account was stolen. Labelling is expensive and often wrong,
and a model can only ever be as good as the labels it was taught from. Keep that thought — it
comes back on Day 3 when we look at **poisoning** the labels on purpose (§3.3).

In [ ]:
# ---- The LABEL: the thing we want the model to predict ----------------------------
# .astype(int) turns True/False into 1/0, because the model wants numbers.
label = logs["Is Attack IP"].astype(int)

# ---- How lopsided is this problem? --------------------------------------------------
# .value_counts() counts how many times each value appears.
counts = label.value_counts().rename({0: "benign", 1: "attack"})
print("How many of each?")
print(counts)

# ---- Why this lopsidedness matters (this is the section 2.7 warning, made real) -------
share_benign = (label == 0).mean()      # .mean() of True/False gives the FRACTION that are True
print("\nBenign logins are {:.1%} of the data.".format(share_benign))
print("So a lazy model that just says 'benign' every single time would score")
print("{:.1%} ACCURACY -- while catching exactly zero attacks.".format(share_benign))
print("\nThat is why we will judge this model on PRECISION and RECALL, never accuracy.")

**What you should see:** about **8,238 benign** and **707 attack** rows — attacks are roughly
**8%** of the log. A do-nothing model scores about **92% accuracy**. Remember that number: when
our real model reports its score, 92% is the bar that means *"learned nothing at all."*

## Step 4 — Feature engineering: turn raw log lines into numbers

A model cannot read `"Mozilla/5.0 (X11; Linux armv7l) AppleWebKit/537.36..."`. It only
understands **numbers**. Turning raw records into useful numeric columns is called
**feature engineering**, and it is where most of the human skill in machine learning lives.

We build **five** features. Each one is a signal a human analyst would also look at:

| Feature | What it captures | Why it might reveal an attack |
|---|---|---|
| `login_successful` | did the login succeed? | password-guessing fails far more often than it succeeds |
| `is_scripted_ua` | is the client a script (`curl`, `python-requests`)? | humans use browsers; bots use scripts |
| `is_foreign` | is the source country not the usual one (Norway)? | logins from abroad are riskier for this app |
| `hour` | hour of the day, 0–23 | 3 a.m. logins are more suspicious than 3 p.m. ones |
| `ip_fail_count` | how many failed logins came from this IP | a brute-force IP fails over and over |

Four of those five look **only at the single log line in front of them**. The fifth,
`ip_fail_count`, is different: to work it out you have to look at **other rows**. That turns
out to matter a lot, so we build the four easy ones now and come back to the fifth in Step 6.

In [ ]:
# ---- Prepare the user-agent text -------------------------------------------------
# .fillna("") replaces any blank user-agent with an empty piece of text, so the search
# below cannot crash. .str.lower() makes it all lower-case so our keyword search is
# case-insensitive ("Python-Requests" and "python-requests" both match).
ua_text = logs["User Agent String"].fillna("").str.lower()

# ---- Build the four "one row at a time" features ------------------------------------
# A DataFrame built from a { } dictionary: each "name": values pair becomes a column.
features = pd.DataFrame({

    # Did it work? 1 = success, 0 = failure.
    "login_successful": logs["Login Successful"].astype(int),

    # Does the user-agent name a scripting tool? The | means OR, so this asks
    # "does the text contain python-requests OR curl OR go-http OR ...".
    # .str.contains(...) gives True/False; .astype(int) turns that into 1/0.
    "is_scripted_ua": ua_text.str.contains(
        "python-requests|curl|go-http|wget|scrapy|okhttp").astype(int),

    # Is the country anything other than Norway ("NO"), this app's home country?
    # != means "is not equal to".
    "is_foreign": (logs["Country"] != "NO").astype(int),

    # .dt.hour pulls just the hour out of a full date-and-time value.
    "hour": logs["Login Timestamp"].dt.hour,
})

print("Four features built. The first five rows:")
print(features.head())

print("\nHow common is each signal?")
for column in ["login_successful", "is_scripted_ua", "is_foreign"]:
    print("  {:<18} is 1 in {:>6.1%} of logins".format(column, features[column].mean()))

**What you should see:** a small table of 1s, 0s and hours, then three percentages. Note how
rare `is_scripted_ua` is and how common `is_foreign` is — that difference comes back to bite us
in Step 12.

## Step 5 — Split the log **before** we do anything else

Same rule as Lab 3b, and it is the most important habit in this whole course:

- The model may only **learn** from the training rows.
- We grade it on **test** rows it has never seen.

We split **75% / 25%** here. One addition compared with Lab 3b: `stratify`. Because attacks are
only ~8% of the data, a careless random split could hand the test set far too few attacks to
judge anything. `stratify=label` forces the same attack-to-benign ratio into both halves.

We split the **row numbers** rather than the features, because in the next step we still have
one more feature to build — and it must be built from the training rows only.

In [ ]:
# ---- Decide which rows are for learning and which are held back ---------------------
# logs.index is just the list of row numbers. We split THAT, so we can use the same
# split for anything we build later.
# random_state=42 fixes WHICH rows are held back, so everyone in the room gets
# identical numbers.
train_rows, test_rows = train_test_split(
    logs.index,
    test_size=0.25,        # 25% held back for the exam
    random_state=42,       # same split for everyone, every time
    stratify=label,        # keep the attack/benign ratio identical in both halves
)

print("Rows the model may learn from (training set): {:,}".format(len(train_rows)))
print("Rows held back for the exam   (test set)    : {:,}".format(len(test_rows)))
print("\nAttacks in the training set: {:,}".format(int(label.loc[train_rows].sum())))
print("Attacks in the test set    : {:,}".format(int(label.loc[test_rows].sum())))

**What you should see:** **6,708** training rows and **2,237** test rows, holding **530** and
**177** attacks respectively. Both work out at about 7.9% attacks — that is `stratify` doing its
job.

## Step 6 — The fifth feature, built without cheating

`ip_fail_count` answers: *how many failed logins came from this IP address?* To work that out
for one row, you have to count **other** rows.

Here is the trap. If we count failures across the **whole** log, then a number derived from the
test rows leaks into the training data. The model gets a peek at the exam paper, its score comes
out flattering, and it will disappoint you in production. This is called **data leakage**, and
it is one of the most common ways real machine-learning projects quietly fail.

It is the exact same rule you already met in Lab 3b, where we fitted the scaler on training data
only. Say it once and it covers both:

> **Anything a feature learns by looking across rows must be learned from the training rows
> only, then applied to both halves.**

In [ ]:
# ---- Count failures per IP -- using TRAINING ROWS ONLY -------------------------------
# Read this line inside-out:
#   logs.loc[train_rows]                    -> just the training rows
#   [... "Login Successful" == False]       -> of those, just the FAILED logins
#   .groupby("IP Address").size()           -> count how many per IP address
train_logs = logs.loc[train_rows]
fails_per_ip = (train_logs.loc[train_logs["Login Successful"] == False]
                .groupby("IP Address")
                .size())

print("We learned a failure count for {:,} distinct IP addresses.".format(len(fails_per_ip)))
print("The busiest failing IPs in the training data:")
print(fails_per_ip.sort_values(ascending=False).head(5))

# ---- Apply that lookup table to EVERY row, training and test alike --------------------
# .map() looks each row's IP up in the table we just built.
# An IP that never appears in the training data gets a blank, so .fillna(0) makes it 0 --
# which is honest: as far as our model knows, that IP has no failure history.
features["ip_fail_count"] = (logs["IP Address"]
                             .map(fails_per_ip)
                             .fillna(0)
                             .astype(int))

# ---- Now cut the finished feature table into the two halves ---------------------------
X_train = features.loc[train_rows]
X_test  = features.loc[test_rows]
y_train = label.loc[train_rows]
y_test  = label.loc[test_rows]

print("\nAll five features are ready.")
print("Training features: {} rows x {} columns".format(*X_train.shape))
print("Test features    : {} rows x {} columns".format(*X_test.shape))
print("\nA peek at the finished feature table:")
print(X_train.head())

**What you should see:** a handful of IPs with very high failure counts — those are the
brute-force sources you found by hand in Lab 3 — and then a five-column feature table,
**6,708 × 5** for training and **2,237 × 5** for the test.

## Step 7 — Normalize to 0–1 (and an honest note)

We put every column on the 0–1 ruler with `MinMaxScaler`, exactly as in Lab 3b.

**Honest caveat, because this course does not hand-wave:** a Decision Tree **does not need
normalizing**. It looks at one feature at a time and asks "is this value above or below some
cut-off?", so the size of the ruler makes no difference to it whatsoever. Lab 3b's KNN cared
enormously because it measured distances across all features at once; a tree simply doesn't.

So why scale at all?

1. Because the **neural network in §2.8.4** trains on this exact same prepared data, and it
   *does* need it.
2. Because building one clean pipeline you can hand to any model is good practice.

We will actually **prove** the tree doesn't care, in the next step.

In [ ]:
# ---- Learn the ruler from TRAINING data only (the Lab 3b golden rule) -------------
# .fit() looks at the training rows and remembers each column's min and max.
scaler = MinMaxScaler().fit(X_train)

# ---- Apply that same remembered ruler to both halves --------------------------------
# .transform() rewrites each value as (value - min) / (max - min).
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# transform() hands back a bare grid of numbers, so we wrap it into a table again
# purely so it prints with proper column names.
print("Before scaling (raw units) -- look at the ranges:")
print(X_train.agg(["min", "max"]))

print("\nAfter scaling (everything squeezed onto a 0-1 ruler):")
print(pd.DataFrame(X_train_scaled, columns=X_train.columns).agg(["min", "max"]).round(2))

**What you should see:** `hour` running 0–23 and `ip_fail_count` running into the hundreds
before scaling; every column running 0.0–1.0 afterwards.

## Step 8 — Train the tree, then **draw** it

`fit()` is the line where learning happens. Give the tree the training features and the training
labels, and it works out its own flow-chart of questions.

`max_depth=4` limits it to four questions deep. Left unlimited, a tree will keep asking
questions until it has memorised every single training row — which scores beautifully on data it
has seen and terribly on anything new. That failure has a name: **overfitting**. A depth limit
is the simplest cure, and it has the happy side effect of keeping the tree small enough to read.

### Two trees, one model

We fit the tree **twice** — once on the scaled features and once on the raw ones — for a reason
that pays off immediately:

- The **scaled** tree is the one we score, so it matches the pipeline the neural network will use.
- The **raw** tree is the one we *draw*, because its questions read like `hour <= 11.5` instead
  of the meaningless `hour <= 0.46`.

They are the same model. We won't just claim that — the cell below **checks** that both trees
make identical predictions on all 2,237 test rows, and says so out loud. That check is also the
proof of the Step 7 claim that a tree doesn't care about scaling.

In [ ]:
# ---- Train the tree we will SCORE (on the scaled features) --------------------------
# random_state=42 makes the training repeatable, so your tree matches everyone else's.
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train_scaled, y_train)

# ---- Train an identical twin on the RAW features, purely so the picture is readable ---
tree_readable = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_readable.fit(X_train, y_train)

# ---- Prove they are the same model ---------------------------------------------------
# (a == b).all() asks "is every single one of these comparisons True?"
same = (tree.predict(X_test_scaled) == tree_readable.predict(X_test)).all()
print("Scaled tree and raw tree agree on all {:,} test rows: {}".format(len(X_test), same))
print("-> Confirmed: scaling makes no difference to a Decision Tree.\n")

# ---- A reusable drawing helper -------------------------------------------------------
#
# CALL THIS AFTER ANY fit() AND IT DRAWS WHATEVER MODEL YOU HAND IT.
# Change the model, call it again, and you get the new flow-chart -- there is nothing
# to keep in sync by hand.
#
def show_tree(model, title, depth_to_show=3):
    """Draw a trained Decision Tree as a flow-chart, and print it as text underneath."""
    plt.figure(figsize=(17, 8))
    plot_tree(
        model,
        feature_names=list(X_train.columns),      # label the questions with real column names
        class_names=["benign", "attack"],         # label the answers in English
        filled=True,        # colour each box by which answer it leans towards
        rounded=True,       # cosmetic: rounded corners
        impurity=False,     # hide the "gini" maths -- not needed to read the chart
        proportion=True,    # show shares rather than raw counts, easier to compare
        fontsize=9,
        max_depth=depth_to_show,   # only DRAW this many levels, so it stays legible
    )
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.show()

    # The same flow-chart as plain text -- handy for notes, and it always renders.
    print("\nThe same tree, as text:\n")
    print(export_text(model, feature_names=list(X_train.columns),
                      max_depth=depth_to_show, show_weights=False))


# ---- Draw the tree the model actually learned -----------------------------------------
show_tree(tree_readable, "Decision Tree (default) - the flow-chart it learned")

**What you should see:** first the confirmation line that both trees agree on every test row,
then a colour-coded flow-chart.

### How to read the picture

- **Read it top-down.** Start at the box at the top and work downwards.
- **Each box's first line is a question** — for example `is_foreign <= 0.5`, which is just a
  clumsy way of writing *"is this login NOT foreign?"* (`is_foreign` is 0 or 1, so "0.5 or less"
  means "it's a 0").
- **Go left for True, right for False.** The top two arrows are labelled so, and every fork below
  follows the same convention.
- **`samples`** = what share of the training logins reach this box.
- **`value`** = how those logins split, as `[benign, attack]`.
- **`class`** = the answer this box would give if you stopped here.
- **Colour** = the same thing at a glance: **orange leans benign, blue leans attack**, and the
  stronger the colour the more one-sided the box.
- The grey `(...)` boxes along the bottom are branches we asked it not to draw, to keep the
  picture legible. `depth_to_show` controls that.

Take a moment on this picture — **nobody wrote those questions.** The tree chose the feature and
the cut-off at every single box by itself, from the labelled examples. That is the difference
between Lab 3 and Lab 3c in one image.

Now count the blue boxes. There is **one**: foreign *and* scripted, reached by 0.3% of logins.
Everything else in the chart says "benign". You have just read, off the picture, exactly why the
score in the next step is going to be what it is.

In [ ]:
# ---- Which features did the tree actually lean on? -------------------------------------
# A tree can tell us how much each feature contributed to its decisions.
# The numbers are shares and add up to 1.0.
importance = (pd.Series(tree.feature_importances_, index=X_train.columns)
              .sort_values(ascending=False))

print("What the tree relied on (feature importance):")
for name, value in importance.items():
    bar = "#" * int(round(value * 40))      # a crude bar chart made of # characters
    print("  {:<18} {:>5.1%}  {}".format(name, value, bar))

**What you should see:** `is_foreign` well out in front, then `is_scripted_ua` and
`ip_fail_count`, with `hour` and `login_successful` barely used. Hold on to that ranking — Step
12 is about what it means.

## Step 9 — Score it on the untouched test rows

The test set is the locked-away final exam: 2,237 logins the tree has never seen. We ask it to
judge every one, then lay the results out in the **confusion matrix** from §2.7:

|  | model says benign | model says attack |
|---|---|---|
| **really benign** | TN — correctly ignored | **FP** — false alarm |
| **really attack** | **FN** — missed attack | TP — caught it |

The two mistakes cost very different things. An **FP** wastes an analyst's afternoon. An **FN**
is a breach nobody noticed.

In [ ]:
# ---- Ask the tree to judge every held-back login ---------------------------------------
y_pred = tree.predict(X_test_scaled)

# ---- Build and draw the confusion matrix -------------------------------------------------
matrix = confusion_matrix(y_test, y_pred, labels=[0, 1])

display = ConfusionMatrixDisplay(confusion_matrix=matrix,
                                 display_labels=["benign", "attack"])
display.plot(cmap="Blues", colorbar=False)
plt.title("Decision Tree (default) - test-set confusion matrix")
plt.show()

# ---- Read the four boxes out in plain English ---------------------------------------------
# .ravel() flattens the 2x2 grid into four values, in this order.
tn, fp, fn, tp = matrix.ravel()
print("TP  attacks caught      : {:>5}".format(tp))
print("FN  attacks MISSED      : {:>5}   <- the expensive mistake".format(fn))
print("FP  false alarms        : {:>5}".format(fp))
print("TN  benign left alone   : {:>5}".format(tn))

# zero_division=0 just stops a warning if the model never raises a single alert.
print("\nPrecision (of our alerts, how many were real?)   : {:.2f}"
      .format(precision_score(y_test, y_pred, zero_division=0)))
print("Recall    (of real attacks, how many did we catch?): {:.2f}"
      .format(recall_score(y_test, y_pred)))

**What you should see:** a nearly perfect **precision** (about 1.00) and a dismal **recall**
(about 0.04) — roughly **7 attacks caught out of 177**, with almost no false alarms.

The tree has played it safe. Because benign logins outnumber attacks eleven to one, the cheapest
way for it to be "right" is to stay quiet and only shout when it is certain. It is technically
accurate about 92% of the time — and, as Step 3 warned, that is exactly the score for learning
nothing.

Does that behaviour sound familiar? It should. Let's put it next to your Lab 3 rules.

## Step 10 — Head-to-head against your Lab 3 hand-written rules

Now the comparison this whole day has been building towards. We rebuild a simple version of your
Lab 3 rules — *flag anything scripted, or any IP with more than 5 failed logins* — and score it
on **exactly the same** 2,237 test rows, with exactly the same scoreboard.

In [ ]:
# ---- Your Lab 3 rules, written out on the test rows ---------------------------------
# The | means OR. So: flag it if the client is a script, OR if its IP has failed a lot.
rule_pred = ((X_test["is_scripted_ua"] == 1) | (X_test["ip_fail_count"] > 5)).astype(int)


# ---- One scoring function so every model is judged identically ------------------------
def scoreline(name, predictions):
    """Return one row of the scoreboard for a set of predictions."""
    predictions = pd.Series(np.asarray(predictions), index=y_test.index)
    return {
        "model":          name,
        "precision":      round(precision_score(y_test, predictions, zero_division=0), 2),
        "recall":         round(recall_score(y_test, predictions), 2),
        "attacks_caught": int(((predictions == 1) & (y_test == 1)).sum()),
        "attacks_missed": int(((predictions == 0) & (y_test == 1)).sum()),
        "false_alarms":   int(((predictions == 1) & (y_test == 0)).sum()),
    }


scoreboard = pd.DataFrame([
    scoreline("Lab 3 hand-written rules", rule_pred),
    scoreline("Decision Tree (default)",  y_pred),
])
print(scoreboard.to_string(index=False))

**What you should see:** the two rows are almost identical — both catch about **7 of 177**
attacks. Our shiny learned model has, so far, done no better than the rules you wrote by hand in
an afternoon.

That is a genuinely useful result, and it is not a failure of the notebook. It is the imbalance
problem from §2.7 biting. Let's do something about it.

## Step 11 — Tune the tree, and watch it redraw itself

`class_weight="balanced"` tells the tree: *stop treating a missed attack as cheap.* Attacks are
rare, so each one is scored as proportionally more important — roughly eleven times the weight
of a benign row here. The tree becomes far braver about shouting "attack".

Nothing else changes. Same features, same rows, same depth, same random seed. **Only the price
of a mistake changes.**

Because `show_tree()` draws whatever model you hand it, calling it again on the retrained tree
gives you the new flow-chart automatically — you can see the model's mind change.

In [ ]:
# ---- Same tree, one setting different --------------------------------------------------
tree_balanced = DecisionTreeClassifier(max_depth=4, random_state=42,
                                       class_weight="balanced")
tree_balanced.fit(X_train_scaled, y_train)
y_pred_balanced = tree_balanced.predict(X_test_scaled)

# ---- And its readable twin, for the picture ---------------------------------------------
tree_balanced_readable = DecisionTreeClassifier(max_depth=4, random_state=42,
                                                class_weight="balanced")
tree_balanced_readable.fit(X_train, y_train)

# ---- Redraw. Same helper, different model, new flow-chart. --------------------------------
show_tree(tree_balanced_readable,
          "Decision Tree (BALANCED) - the same helper, a different tree")

**What you should see:** a visibly different flow-chart — more blue, because far more paths now
end in "attack". Compare it with the picture in Step 8: same data, same algorithm, and the model
has reorganised its whole reasoning because we changed what a mistake costs.

One detail worth pointing at: the top box now reads `value = [0.5, 0.5]`, as though attacks and
benign logins were equally common. They aren't — that is the *weighting* talking. `balanced` told
the tree to count each attack about eleven times over, so from the tree's point of view the two
classes now weigh the same.

In [ ]:
# ---- Put all three on one scoreboard ------------------------------------------------------
scoreboard = pd.DataFrame([
    scoreline("Lab 3 hand-written rules", rule_pred),
    scoreline("Decision Tree (default)",  y_pred),
    scoreline("Decision Tree (balanced)", y_pred_balanced),
])
print(scoreboard.to_string(index=False))

# ---- Draw the balanced model's confusion matrix too -----------------------------------------
matrix_b = confusion_matrix(y_test, y_pred_balanced, labels=[0, 1])
ConfusionMatrixDisplay(confusion_matrix=matrix_b,
                       display_labels=["benign", "attack"]).plot(cmap="Blues", colorbar=False)
plt.title("Decision Tree (balanced) - test-set confusion matrix")
plt.show()

**What you should see:** recall leaps from about **0.04 to about 0.89** — from 7 attacks caught
to roughly **158 of 177**. And precision collapses from about 1.00 to about **0.19**: the model
now raises around **670 false alarms**.

**This is the §2.7 precision/recall trade-off, live and in your own hands.** There is no setting
that gives you both. You choose:

- **Incident response, fraud, account takeover** → favour **recall**. Missing a real breach is
  the unaffordable mistake; you will pay for it with analyst hours.
- **High-volume triage with a small team** → favour **precision**. A queue of 670 junk alerts is
  a queue nobody reads, and then you miss things anyway.

That choice is a business decision wearing a technical costume. It belongs to the SOC lead, not
to the model.

In [ ]:
# ---- See the trade-off as a picture --------------------------------------------------------
positions = np.arange(len(scoreboard))     # 0, 1, 2 -- one slot per model
bar_width = 0.38

plt.figure(figsize=(9, 4.5))
plt.bar(positions - bar_width/2, scoreboard["precision"], bar_width,
        label="Precision (are our alerts real?)", color="#4C72B0")
plt.bar(positions + bar_width/2, scoreboard["recall"], bar_width,
        label="Recall (are we catching attacks?)", color="#DD8452")

plt.xticks(positions, ["Lab 3\nrules", "Tree\n(default)", "Tree\n(balanced)"])
plt.ylim(0, 1.05)
plt.ylabel("score  (1.0 is perfect)")
plt.title("You do not get both: precision vs recall")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

**What you should see:** the two bars trading places as you move right. That seesaw *is* the
trade-off.

## Step 12 — Read the tree honestly: what did it *actually* learn?

Recall 0.89 looks like a triumph. Before you take it to your boss, **go back and look at the
flow-chart in Step 11.**

The very first question — the one the tree considered most valuable of all — is `is_foreign`.
And look what happens at that fork: the whole **right-hand branch turns blue at once**. Answer
"not Norwegian" and you are already most of the way to being flagged; almost everything below
that point is detail. The feature-importance list says the same thing in numbers.

Which raises an uncomfortable question: has this model learned anything clever at all, or has it
just learned **"foreign login = attack"**?

There is a way to find out. Write that one-line rule out by hand — no model, no training, no
maths — and score it on the same test rows.

In [ ]:
# ---- The dumbest possible rule: flag every non-Norwegian login --------------------------
naive_pred = (X_test["is_foreign"] == 1).astype(int)

comparison = pd.DataFrame([
    scoreline("Tree (balanced)",            y_pred_balanced),
    scoreline("One line: 'foreign = attack'", naive_pred),
])
print(comparison.to_string(index=False))

**What you should see:** the one-line rule scores **essentially the same as the trained model**.
Not similar — the same. It catches the **identical 158 attacks**, misses the identical 19, and
its only cost is about 27 extra false alarms out of ~670.

Sit with that for a second. We engineered five features, split the data properly, avoided
leakage, normalized, trained a model and tuned it — and it found a rule a bored analyst would
have written in ten seconds.

### So was the lab a waste of time? No. This *is* the lesson.

Three things just happened, and all three matter more than the recall number:

1. **The score did not tell you this. The picture did.** A recall of 0.89 on a slide looks like
   success. Only by *reading the model* did you find out it rests on one crude signal. That is
   the entire argument for glass-box models, and it is why §2.10 on explainability exists.

2. **The model is only as good as its features.** The tree could not learn anything subtler than
   `is_foreign`, because we never gave it anything subtler. It cannot see that an account
   normally logs in from Oslo and suddenly appeared in three countries in an hour — we never
   built that feature. Machine learning does not rescue you from knowing your data; it rewards
   you for knowing it.

3. **It would fail the moment the world changed.** Hire a remote team in Spain and this model
   alerts on every one of them. An attacker who rents a Norwegian VPS walks straight past it.
   The rule looked strong only because, *in this particular log*, most attacks happened to come
   from abroad. That is called learning a **spurious correlation** — and it is one of the most
   common reasons a model that dazzles in a demo dies in production.

> **The takeaway to write down:** a model that scores well is not the same as a model that has
> learned the right thing. Always ask what it is keying on. If you cannot ask, you are trusting
> something you cannot interrogate.

**What you could try next, if you have time:** add a feature the tree cannot cheat with — for
example, how many *distinct countries* an account logged in from in a single day, or how far
this login's hour is from that account's normal hour. Then rerun from Step 6 and call
`show_tree()` again. If the root question stops being `is_foreign`, you have taught it something
real.

## Wrap-up

What you did in this lab:

- Ran the **full pipeline** on real security data: *engineer features → split → normalize →
  train → score*, the Labs 3a/3b skills applied to the log from Labs 2 and 3.
- Used the answer key as a **teacher** rather than a marking scheme — and saw why "where do the
  labels come from?" is the hard question in real security ML.
- Avoided **leakage** twice: once fitting the scaler, once counting failures per IP.
- Graded a real model with the **§2.7 confusion matrix**, on predictions it had never seen.
- Watched the **precision/recall trade-off** move under your hands with a single setting.
- **Read the model** and discovered that a good score can hide a shallow rule.

Where this goes next:

- **§2.8.4 — neural networks.** The same five features, the same split, a model whose reasoning
  you *cannot* read like this. Now you know exactly what you are giving up.
- **§2.10 — explainable AI.** Today's flow-chart was the easy case. What do you do when the
  model has no flow-chart to draw?
- **Day 3 (§3.3) — poisoning.** If the labels teach the model, what happens when someone
  deliberately teaches it wrong?